In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Setup and Installations
First, we will install the `faker` library to generate authentic-sounding Indian names, and set up our environment.

In [2]:
!pip install -q faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.1 MB/s eta 0:00:0000:0100:01


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from faker import Faker
import random
import numpy as np

# Set random seeds to ensure our results are reproducible
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Check for GPU availability. Using a GPU is highly recommended for 30,000 epochs!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## TASK-0: DATASET (Using Faker)
We generate 1000 Indian first names using `Faker('en_IN')` to act as our training set.

In [4]:
# Initialize Faker for Indian names
fake = Faker('en_IN')
generated_names = set()
TARGET_DATASET_SIZE = 10000

print(f"Generating up to {TARGET_DATASET_SIZE} unique names... Please wait.")

# Phase 1: Extract real Indian names from Faker (with a safety break)
attempts = 0
while len(generated_names) < 4000 and attempts < 15000:
    name = fake.first_name()
    # Keep only alphabetical names of reasonable length to keep vocab clean
    if len(name) >= 3 and name.isalpha():
        generated_names.add(name.capitalize())
    attempts += 1

print(f"Gathered {len(generated_names)} unique names from Faker.")

# Phase 2: Procedural Generation to fill the rest of the dataset
prefixes = ["Aar", "Vi", "Riy", "Kav", "Ish", "An", "Sneh", "Raj", "Shr", "Dhruv", "Ne", "Poo", "Gaur", "Sam", "Tany", "Tar", "Man", "Yash", "Krit", "Dev", "Rahul", "Amit", "Suni", "Ravi", "Vik", "Pran", "Adit", "Kart", "Deep", "Shiv", "Har", "Meera", "Moh", "Sur", "Nir", "Par", "Bhav", "Lak"]
middles = ["", "a", "i", "u", "ee", "endra", "esh", "an"]
suffixes = ["av", "an", "a", "ya", "ika", "ish", "al", "esh", "uti", "it", "ha", "ja", "ir", "un", "ansh", "in", "ita", "il", "ram", "ji", "preet", "kar", "eshwar", "ak", "deep", "nath", "kant"]
ends = ["", "n", "m", "s", "th", "r", "l", "sh", "k"]

# Safety break to prevent infinite loops if combinations run out
procedural_attempts = 0
max_procedural_attempts = 50000

while len(generated_names) < TARGET_DATASET_SIZE and procedural_attempts < max_procedural_attempts:
    # Randomly assemble syllables
    name = random.choice(prefixes) + random.choice(middles) + random.choice(suffixes) + random.choice(ends)
    if len(name) >= 3:
        generated_names.add(name.capitalize())
    procedural_attempts += 1

# Convert our set to a list and shuffle it
names_list = list(generated_names)[:TARGET_DATASET_SIZE]
random.shuffle(names_list)

# Save to text file
with open("TrainingNames.txt", "w") as f:
    for name in names_list:
        f.write(name + "\n")

print(f"Successfully generated and saved {len(names_list)} names to TrainingNames.txt")
print(f"SAMPLE OF TRAINING DATA: {names_list[:15]}")

Generating up to 10000 unique names... Please wait.
Gathered 530 unique names from Faker.
Successfully generated and saved 10000 names to TrainingNames.txt
SAMPLE OF TRAINING DATA: ['Bhaveshunm', 'Bhavuak', 'Ganga', 'Bhavanikar', 'Devapreetl', 'Samayam', 'Shivendranathth', 'Kritendraishs', 'Nirayak', 'Yashanitam', 'Bhaveeishs', 'Deveeunn', 'Rahulueshwarsh', 'Haranilm', 'Kartendrajith']


## Data Preprocessing
Building the character-level vocabulary and PyTorch Dataset/DataLoader.

In [5]:
# 1. Read the newly generated names and force them to UPPERCASE
with open("TrainingNames.txt", "r") as f:
    names = [line.strip().upper() for line in f.readlines()]

# 2. Build Vocabulary dynamically (Now only uppercase letters + tokens)
chars = set("".join(names))
vocab = ['<PAD>', '<SOS>', '<EOS>'] + sorted(list(chars))
char_to_ix = {ch: i for i, ch in enumerate(vocab)}
ix_to_char = {i: ch for i, ch in enumerate(vocab)}
vocab_size = len(vocab)

# 3. Create the Dataset Class
class NameDataset(Dataset):
    def __init__(self, names, char_to_ix):
        self.names = names
        self.char_to_ix = char_to_ix
        
    def __len__(self):
        return len(self.names)
    
    def __getitem__(self, idx):
        name = self.names[idx]
        x_str = ['<SOS>'] + list(name)
        y_str = list(name) + ['<EOS>']
        
        x = [self.char_to_ix[ch] for ch in x_str]
        y = [self.char_to_ix[ch] for ch in y_str]
        return torch.tensor(x), torch.tensor(y)

# 4. Collate Function
def collate_fn(batch):
    xs, ys = zip(*batch)
    max_len = max([len(x) for x in xs])
    
    padded_x = torch.full((len(xs), max_len), char_to_ix['<PAD>'], dtype=torch.long)
    padded_y = torch.full((len(ys), max_len), char_to_ix['<PAD>'], dtype=torch.long)
    
    for i, (x, y) in enumerate(zip(xs, ys)):
        padded_x[i, :len(x)] = x
        padded_y[i, :len(y)] = y
        
    return padded_x, padded_y

# 5. Initialize Dataset and DataLoader
dataset = NameDataset(names, char_to_ix)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)

print(f"Total Names Loaded: {len(names)}")
print(f"Vocabulary Size: {vocab_size}")
print(f"Characters in Vocab: {vocab}")

Total Names Loaded: 10000
Vocabulary Size: 29
Characters in Vocab: ['<PAD>', '<SOS>', '<EOS>', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


## TASK-1: MODEL IMPLEMENTATIONS
Vanilla RNN, BLSTM, and RNN with Basic Attention.

In [6]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(VanillaRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=char_to_ix['<PAD>'])
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden=None):
        out = self.embedding(x)
        out, hidden = self.rnn(out, hidden)
        return self.fc(out), hidden

class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(BLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=char_to_ix['<PAD>'])
        # Setting bidirectional=True creates two hidden states per layer
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        # Multiply hidden_size by 2 to account for both forward and backward states
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        
    def forward(self, x, hidden=None):
        out = self.embedding(x)
        out, hidden = self.lstm(out, hidden)
        return self.fc(out), hidden

class RNNAttention(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(RNNAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=char_to_ix['<PAD>'])
        self.rnn = nn.RNN(embed_size, hidden_size, num_layers, batch_first=True)
        
        # Simple self-attention weights
        self.attention_weights = nn.Linear(hidden_size, 1)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)
        
    def forward(self, x, hidden=None):
        embeds = self.embedding(x)
        rnn_out, hidden = self.rnn(embeds, hidden) 
        
        # Calculate attention scores and weights
        attn_scores = self.attention_weights(rnn_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        
        # Apply weights to RNN outputs to create the context vector
        context = rnn_out * attn_weights
        
        # Concatenate context with the original RNN output
        combined = torch.cat((rnn_out, context), dim=2)
        return self.fc(combined), hidden

## Training Setup (3000 Epochs)
Training the models. Because of the massive epoch count, the console will only output loss every 300 epochs.

In [7]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_model(model, dataloader, epochs, lr=0.001):
    model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_ix['<PAD>'])
    # Lowered learning rate for stability
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            output, _ = model(x)
            
            output = output.view(-1, vocab_size)
            y = y.view(-1)
            
            loss = criterion(output, y)
            loss.backward()
            
            # CRITICAL ADDITION: Gradient Clipping prevents exploding gradients in RNNs
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
            
            optimizer.step()
            epoch_loss += loss.item()
            
        # Print loss every 10 epochs
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(dataloader):.4f}")
            
    return model

# --- TUNED HYPERPARAMETERS ---
EMBED_SIZE = 128     # Increased from 64
HIDDEN_SIZE = 256    # Increased from 128
NUM_LAYERS = 2       # Increased from 1 for deeper learning capacity
LEARNING_RATE = 0.001 # Decreased from 0.005 to prevent bouncing loss
EPOCHS = 200         # Sane epoch count

print("--- Training Vanilla RNN ---")
rnn_model = train_model(VanillaRNN(vocab_size, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS), dataloader, EPOCHS, LEARNING_RATE)

print("\n--- Training BLSTM ---")
blstm_model = train_model(BLSTM(vocab_size, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS), dataloader, EPOCHS, LEARNING_RATE)

print("\n--- Training RNN with Attention ---")
attn_model = train_model(RNNAttention(vocab_size, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS), dataloader, EPOCHS, LEARNING_RATE)

--- Training Vanilla RNN ---
Epoch 1/200 | Loss: 2.0125
Epoch 10/200 | Loss: 1.1738
Epoch 20/200 | Loss: 1.1484
Epoch 30/200 | Loss: 1.1287
Epoch 40/200 | Loss: 1.1098
Epoch 50/200 | Loss: 1.0914
Epoch 60/200 | Loss: 1.0739
Epoch 70/200 | Loss: 1.0500
Epoch 80/200 | Loss: 1.0343
Epoch 90/200 | Loss: 1.0180
Epoch 100/200 | Loss: 1.0001
Epoch 110/200 | Loss: 0.9939
Epoch 120/200 | Loss: 0.9813
Epoch 130/200 | Loss: 0.9798
Epoch 140/200 | Loss: 0.9766
Epoch 150/200 | Loss: 0.9682
Epoch 160/200 | Loss: 0.9648
Epoch 170/200 | Loss: 0.9622
Epoch 180/200 | Loss: 0.9658
Epoch 190/200 | Loss: 0.9584
Epoch 200/200 | Loss: 0.9577

--- Training BLSTM ---
Epoch 1/200 | Loss: 1.5034
Epoch 10/200 | Loss: 0.0008
Epoch 20/200 | Loss: 0.0002
Epoch 30/200 | Loss: 0.0001
Epoch 40/200 | Loss: 0.0001
Epoch 50/200 | Loss: 0.0000
Epoch 60/200 | Loss: 0.0000
Epoch 70/200 | Loss: 0.0000
Epoch 80/200 | Loss: 0.0000
Epoch 90/200 | Loss: 0.0000
Epoch 100/200 | Loss: 0.0000
Epoch 110/200 | Loss: 0.0000
Epoch 120/20

## Generation Setup
We force the sequence to begin with `<SOS>` + a random uppercase letter from the vocabulary. This injects initial randomness to force novelty.

In [13]:
def generate_names_random_start(model, num_names=100, max_len=15, temperature=1.0, min_len=4):
    model.eval()
    generated = []
    
    # Valid starting characters (no special tokens)
    valid_start_chars = [ch for ch in vocab if ch not in ['<PAD>', '<SOS>', '<EOS>']]
    eos_idx = char_to_ix['<EOS>']
    
    with torch.no_grad():
        for _ in range(num_names):
            start_char = random.choice(valid_start_chars)
            current_seq = [char_to_ix['<SOS>'], char_to_ix[start_char]]
            name = start_char
            
            for _ in range(max_len - 1):
                x = torch.tensor([current_seq]).to(device)
                
                # Feed the sequence generated so far
                output, _ = model(x, hidden=None)
                
                # Get logits for the last time step
                logits = output[0, -1, :] / temperature
                
                # THE HACK: If the name is shorter than min_len, ban the <EOS> token
                # by setting its logit to negative infinity.
                if len(name) < min_len:
                    logits[eos_idx] = -float('Inf')
                
                probs = torch.softmax(logits, dim=0).cpu().numpy()
                
                # Sample the next character
                next_char_idx = np.random.choice(len(vocab), p=probs)
                next_char = ix_to_char[next_char_idx]
                
                if next_char == '<EOS>':
                    break
                if next_char not in ['<PAD>', '<SOS>']:
                    name += next_char
                    
                current_seq.append(next_char_idx)
                
            if len(name) > 2: 
                generated.append(name.capitalize())
                
    return generated

print("Generating 500 samples per model (forcing minimum lengths)...")

# We use temperature 0.7 for the normal models
rnn_samples = generate_names_random_start(rnn_model, num_names=500, temperature=0.7)
attn_samples = generate_names_random_start(attn_model, num_names=500, temperature=0.7)

# We use a slightly higher temperature (1.0) for the BLSTM to force it to be 
# more "creative" since it will be confused when denied the <EOS> token.
blstm_samples = generate_names_random_start(blstm_model, num_names=500, temperature=1.0)

print("Samples generated successfully.")

Generating 500 samples per model (forcing minimum lengths)...
Samples generated successfully.


## TASK-2 & 3: EVALUATION & SAMPLES DISPLAY
Let's look at the metrics and directly compare the output. Because of 30,000 epochs, expect the novelty rate to be extremely low unless the random starting character forces it down an unexplored path.

In [14]:
def evaluate_metrics(generated_samples, training_set):
    if not generated_samples: return 0, 0
    
    # Diversity: Unique Generated Names / Total Generated Names
    unique_names = set(generated_samples)
    diversity = len(unique_names) / len(generated_samples)
    
    # Novelty: Generated Names not in Training Set / Total Generated Names
    novel_names = [name for name in unique_names if name not in training_set]
    novelty = len(novel_names) / len(generated_samples)
    
    return novelty, diversity

# Lowercase the training set for fair comparison
train_set_lower = set([n.lower() for n in names])

models_samples = {
    "Vanilla RNN": rnn_samples,
    "BLSTM": blstm_samples,
    "RNN + Attention": attn_samples
}

# --- QUANTITATIVE ANALYSIS ---
print(f"{'Model':<20} | {'Novelty Rate':<15} | {'Diversity Rate':<15}")
print("-" * 55)
for name, samples in models_samples.items():
    novelty, diversity = evaluate_metrics([s.lower() for s in samples], train_set_lower)
    print(f"{name:<20} | {novelty*100:5.2f}%          | {diversity*100:5.2f}%")

# --- QUALITATIVE ANALYSIS ---
print("\n" + "="*70)
print("QUALITATIVE SAMPLES (15 from each model):")
print("="*70)
print(f"Vanilla RNN:     {rnn_samples[:15]}\n")
print(f"BLSTM:           {blstm_samples[:15]}\n")
print(f"RNN + Attention: {attn_samples[:15]}\n")

Model                | Novelty Rate    | Diversity Rate 
-------------------------------------------------------
Vanilla RNN          |  7.40%          | 76.00%
BLSTM                | 87.80%          | 87.80%
RNN + Attention      |  6.60%          | 69.60%

QUALITATIVE SAMPLES (15 from each model):
Vanilla RNN:     ['Hareshinn', 'Meeraanakl', 'Yasheshilk', 'Harendrajam', 'Kartendraal', 'Samupreetr', 'Charan', 'Yasheeyak', 'Vikanyas', 'Zansi', 'Zayan', 'Charles', 'Jackson', 'Warinder', 'Ekanta']

BLSTM:           ['Adfw', 'Wwww', 'Vjjj', 'Zcjj', 'Bdes', 'Ilde', 'Ghju', 'Pooj', 'Krus', 'Urlm', 'Gwmm', 'Xecu', 'Ddww', 'Kmmu', 'Owww']

RNN + Attention: ['Januja', 'Qarin', 'Yashendrairk', 'Aareshinn', 'Lakeshm', 'Manuyam', 'Tanyeeeshr', 'Utansh', 'Jason', 'Ekapad', 'Dakshesh', 'Dalaja', 'Xavier', 'Qasim', 'Parendrakarn']

